# Affective Priming (J2) — Personal Control Domain — Vector Injection vs. Real Prompt Simulation

Full pipeline, consolidating everything decided in this project:

**Phase 1**: extract a lack-of-control↔in-control direction from the real Mega Study `c_writing` data (meandiff + probe, `extraction_tokens='all'`).

**Phase 2**: two arms, run across the same set of personas (`persona_summary`, live from the Twin-2K-500 dataset):
- **Simulation arm**: real system instruction + persona + real writing-task instruction (QID31 prime/lack-of-control / QID32 control/in-control) + real downstream questions (manip check / fatigue / predictability), ONE combined generation producing Q1(writing)+Q2+Q3+Q4 — this replicates the actual J2 digital-twin methodology (no injection).
- **Injection arm**: same persona, but Q1 (writing) is skipped entirely; the lack-of-control↔in-control vector is injected at every generation step (positive β = toward lack-of-control, negative β = toward in-control, same vector, sign flipped) while generating Q2+Q3+Q4 only.

JSON output is constrained via `outlines` (schema-guaranteed), since an 8B model is much less reliable than GPT-4.1 at free-form JSON compliance.

**Caveat**: this notebook has not been run end-to-end on a live GPU session by me (no Colab/GPU access on my end) — it's built directly from the source code and real materials verified earlier in this project, but expect to hit a few bugs on first run. Paste back any errors and they can be fixed.

## 0. Install dependencies

In [ ]:
!git clone https://github.com/aminbana/PsySET.git
%cd PsySET
!pip install -q transformers==4.56.1 transformer_lens==2.16.1
!pip install -q pandas scikit-learn tqdm huggingface_hub requests anthropic

In [ ]:
!pip install -q anthropic

## 1. HuggingFace login
Add your token as a Colab secret named `HUGGINGFACE_TOKEN` first.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')
login(token=HF_TOKEN)

## 2. Load Llama-3.1-8B-Instruct

In [ ]:
import sys
import os
import torch

PSYSET_DIR = "/content/PsySET"
if PSYSET_DIR not in sys.path:
    sys.path.append(PSYSET_DIR)
    sys.path.append(os.path.join(PSYSET_DIR, "LLMs"))
os.chdir(PSYSET_DIR)

from my_llama import SteeredLlamaForCausalLM
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
ASSISTANT_TAG = "<|start_header_id|>assistant<|end_header_id|>\n\n"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = SteeredLlamaForCausalLM.from_pretrained(
    MODEL_NAME, device_map="cuda", torch_dtype=torch.bfloat16
)
model.eval()
print("model loaded:", model.config.model_type, model.config.num_hidden_layers, "layers")

# PHASE 1: Extract the gratitude↔routine vector
## 3. Download the real Mega Study writing data

In [ ]:
import pandas as pd

RESPONSE_CSV_URL = (
    "https://raw.githubusercontent.com/TianyiPeng/Twin-2K-500-Mega-Study/main/"
    ".dat/affective_priming/raw_data/response.csv"
)
RESPONSE_CSV_PATH = "response.csv"
!curl -sL "{RESPONSE_CSV_URL}" -o "{RESPONSE_CSV_PATH}"

_check = pd.read_csv(RESPONSE_CSV_PATH, low_memory=False, skiprows=[1, 2])
print(f"parsed rows: {len(_check)} (expect ~1000)")
print(f"c_writing non-null: {_check['c_writing'].notna().sum()} (expect ~492)")

## 4. Build lack-of-control (prime) vs. in-control (control) contrastive examples

In [ ]:
import numpy as np

QID31_PROMPT = (
    "Please recall a particular incident in which something happened and you did not have "
    "control over the situation. Please describe the situation in which you felt a lack of "
    "control – what happened, how you felt, etc. Spend two minutes detailing this experience "
    "below:"
)
QID32_PROMPT = (
    "Please recall a particular incident in which something happened and you were in control "
    "of the situation. Please describe the situation in which you felt in control – what "
    "happened, how you felt, etc. Spend two minutes detailing this experience below:"
)


def load_megastudy_control(csv_path):
    df = pd.read_csv(csv_path, low_memory=False, skiprows=[1, 2])
    cw = df["c_writing"].astype(str).str.strip()
    valid = cw.notna() & (cw != "") & (cw != "nan")
    df = df[valid].copy()
    df["c_writing"] = cw[valid]
    df = df[df["cond"].isin(["prime", "control"])]

    examples, labels = [], []
    for _, row in df.iterrows():
        if row["cond"] == "prime":
            user_prompt, label = QID31_PROMPT, 1
        else:
            user_prompt, label = QID32_PROMPT, 0
        examples.append({"system": "", "user": user_prompt, "assistant": row["c_writing"]})
        labels.append(label)
    return examples, np.array(labels)


examples, y = load_megastudy_control(RESPONSE_CSV_PATH)
print(f"Total examples: {len(examples)}, lack_of_control(1)={ (y==1).sum() }, in_control(0)={ (y==0).sum() }")

## 5. Tokenize (chat template), extract hidden states across candidate layers

In [ ]:
def to_chat_text(example):
    messages = []
    if example["system"]:
        messages.append({"role": "system", "content": example["system"]})
    messages.append({"role": "user", "content": example["user"]})
    messages.append({"role": "assistant", "content": example["assistant"]})
    return tokenizer.apply_chat_template(messages, tokenize=False)


batch_texts = [to_chat_text(ex) for ex in examples]
print(batch_texts[0][:400])

In [ ]:
from torch.utils.data import DataLoader
from steer_vec_utils import TextDataset, extract_hidden_states, probe_classification

dataset = TextDataset(batch_texts)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)

CANDIDATE_LAYERS = list(range(10, 24))
LOC = 7  # hook_after_mlp_hs (residual stream after MLP)

all_hidden_states = extract_hidden_states(
    dataloader,
    tokenizer,
    model,
    assistant_tag=ASSISTANT_TAG,
    extraction_layers=CANDIDATE_LAYERS,
    extraction_locs=[LOC],
    extraction_tokens="all",  # full prompt+response, matches the real conflated manipulation
    avg_token_dim=True,
    do_final_cat=True,
)
print(all_hidden_states.shape)

## 6. Compute MeanDiff + Probe vectors per layer, pick the best layer

In [ ]:
y_tensor = torch.tensor(y)
results = {}

for i, layer in enumerate(CANDIDATE_LAYERS):
    layer_hs = all_hidden_states[:, i, 0, 0, :].float()  # bfloat16 -> float32 for sklearn

    pos_hs = layer_hs[y == 1]
    neg_hs = layer_hs[y == 0]
    meandiff_vec = pos_hs.mean(dim=0) - neg_hs.mean(dim=0)

    probe_res = probe_classification(layer_hs, y_tensor, fit_intercept=False)

    results[layer] = {
        "meandiff_vec": meandiff_vec,
        "probe_weights": torch.tensor(probe_res["weights"]).flatten().float(),
        "test_acc": float(probe_res["accuracy_test"]),
    }
    print(f"layer {layer}: probe held-out test accuracy = {results[layer]['test_acc']:.3f}")

max_acc = max(results[l]["test_acc"] for l in results)
tied_layers = sorted([l for l in results if results[l]["test_acc"] == max_acc])
BEST_LAYER = tied_layers[len(tied_layers) // 2]  # among ties, pick the middle layer
print(f"Layers tied at max accuracy ({max_acc:.3f}): {tied_layers}")
print(f"BEST_LAYER (middle of tied group): {BEST_LAYER}")

In [ ]:
INJECT_LAYERS = [BEST_LAYER - 1, BEST_LAYER, BEST_LAYER + 1]
INJECT_LAYERS = [l for l in INJECT_LAYERS if l in results]  # 防止超出候选层范围(10-23)

LAYER_VECTORS = {
    layer: (results[layer]["meandiff_vec"] / results[layer]["meandiff_vec"].norm()).to(model.device, dtype=torch.bfloat16)
    for layer in INJECT_LAYERS
}
print(f"注入层: {INJECT_LAYERS}")

In [ ]:
# The vector used for injection (normalized). VECTOR_METHOD: "meandiff" or "probe".
VECTOR_METHOD = "meandiff"
INJECT_LAYER = BEST_LAYER

raw_vec = results[INJECT_LAYER]["meandiff_vec" if VECTOR_METHOD == "meandiff" else "probe_weights"]
CONTROL_VECTOR = (raw_vec / raw_vec.norm()).to(model.device, dtype=torch.bfloat16)
print("CONTROL_VECTOR ready. layer:", INJECT_LAYER, "norm:", CONTROL_VECTOR.norm().item())

# PHASE 2: Simulation arm vs. Injection arm
## 7. Real materials (verified against the Mega Study QSF / GitHub source)

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an AI assistant. Your task is to answer the 'New Survey Question' as if you "
    "are the person described in the 'Persona Profile' (which consists of their past survey "
    "responses). \nRemain consistent with the persona's previous answers and stated traits. "
    "Simulate their responses to new questions while accounting for human cognitive "
    "limitations, uncertainty, and biases.\nFollow all instructions provided for the new "
    "question carefully regarding the format of your answer."
)

COMBINED_PROMPT_HEADER = "## Persona Profile (This individual's past survey responses):\n"
COMBINED_PROMPT_SEPARATOR = (
    "\n\n---\n## New Survey Question & Instructions "
    "(Please respond as the persona described above):\n"
)
GENERAL_LLM_INSTRUCTION = (
    "Please answer the following questions as if you were taking this survey. The expected "
    "output is a JSON object and the format is provided in the end. \n---\n\n"
)

Q_WRITING = "{instruction}\nQuestion Type: Text Entry\nAnswer: [Masked]"

Q_MANIP_CHECK = (
    "Please indicate the extent to which you agree with the following statement regarding how you feel right now.\n"
    "Question Type: Matrix\n"
    "Options:\n  1 = Not at all\n  2 = 2\n  3 = 3\n  4 = 4\n  5 = 5\n  6 = 6\n  7 = Extremely\n"
    "1. I feel in control of my life\nAnswer: [Masked]\n"
    "2. I am free to live my life how I wish\nAnswer: [Masked]\n"
    "3. My experiences in life are due to my own actions\nAnswer: [Masked]"
)

Q_FATIGUE = (
    "Please indicate the extent to which you agree with the following statements relating to your experience reflecting on a recent event earlier in this survey.\n"
    "Question Type: Matrix\n"
    "Options:\n  1 = Not at all\n  2 = 2\n  3 = 3\n  4 = 4\n  5 = 5\n  6 = 6\n  7 = Extremely\n"
    "1. I felt discouraged when recalling and describing the event from my own past\nAnswer: [Masked]\n"
    "2. My mind was running blank when recalling and describing the event from my own past\nAnswer: [Masked]\n"
    "3. I was fully involved in recalling and describing the event\nAnswer: [Masked]\n"
    "4. I found it very hard to think\nAnswer: [Masked]\n"
    "5. It was difficult for me to get motivated to recall and describe the event\nAnswer: [Masked]\n"
    "6. I was distracted and could not focus\nAnswer: [Masked]\n"
    "7. Recalling and describing the event was a pleasant activity\nAnswer: [Masked]\n"
    "8. The task of recalling and describing the event was very interesting\nAnswer: [Masked]\n"
    "9. Very few ideas of a specific event came to my mind\nAnswer: [Masked]"
)

Q_PREDICTABILITY = (
    "At this moment...\n"
    "Question Type: Matrix\n"
    "Options:\n  1 = Completely disagree\n  2 = 2\n  3 = 3\n  4 = 4\n  5 = 5\n  6 = 6\n  7 = Completely agree\n"
    "1. I would like to be around friends who are unpredictable\nAnswer: [Masked]\n"
    "2. I would like to go to places where I have been before so that I know what to expect\nAnswer: [Masked]\n"
    "3. I would not like to go into a situation without knowing what I can expect from it\nAnswer: [Masked]\n"
    "4. I feel that it would be fun to change my plans at the last moment\nAnswer: [Masked]\n"
    "5. I would enjoy the uncertainty of going into a new situation without knowing what might happen\nAnswer: [Masked]\n"
    "6. I would not like to be with people who are capable of unexpected actions\nAnswer: [Masked]\n"
    "7. I would prefer to socialize with familiar friends because I know what to expect from them\nAnswer: [Masked]\n"
    "8. I dislike unpredictable situations\nAnswer: [Masked]"
)

## 8. Structured JSON output via `outlines` (schema-guaranteed, no retry-loop needed)

In [ ]:
import json
import re
import anthropic
from google.colab import userdata

claude_client = anthropic.Anthropic(api_key=userdata.get('CLAUDE_API_KEY'))

PARSE_INSTRUCTION_TEMPLATE = """You will be shown a response from another AI model that was asked to answer survey questions using a 1-7 rating scale.

Q2 (3 items, each rated 1-7): feeling in control of my life, feeling free to live my life how I wish, feeling my experiences are due to my own actions
Q3 (9 items, each rated 1-7): felt discouraged, mind running blank, fully involved, hard to think, hard to get motivated, distracted, pleasant activity, interesting task, few ideas came to mind
Q4 (8 items, each rated 1-7): desire for predictability/familiarity vs. enjoying unpredictability, across 8 statements
{q1_line}

Read the response below and extract the answers. If it clearly answers every item, output ONLY this JSON (no other text):
{{"Q2": [n, n, n], "Q3": [n, n, n, n, n, n, n, n, n], "Q4": [n, n, n, n, n, n, n, n]{q1_key}}}

If any item is missing, unclear, or not actually answered, output exactly:
{{"failed": true}}

Response to parse:
---
{raw_text}
---
"""


def parse_with_claude(raw_text, need_q1=False):
    q1_line = "Q1 (free text): a short first-person written response" if need_q1 else ""
    q1_key = ', "Q1": "..."' if need_q1 else ""
    prompt = PARSE_INSTRUCTION_TEMPLATE.format(q1_line=q1_line, raw_text=raw_text, q1_key=q1_key)

    resp = claude_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    claude_text = resp.content[0].text.strip()
    try:
        parsed = json.loads(claude_text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", claude_text, re.DOTALL)
        if not match:
            return claude_text, None
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            return claude_text, None

    if parsed.get("failed"):
        return claude_text, None

    # 明确校验每个字段的长度对不对，长度不对也算失败
    expected_lengths = {"Q2": 3, "Q3": 9, "Q4": 8}
    for key, length in expected_lengths.items():
        if key not in parsed or not isinstance(parsed[key], list) or len(parsed[key]) != length:
            return claude_text, None
    if need_q1 and "Q1" not in parsed:
        return claude_text, None

    return claude_text, parsed




def generate_and_extract_logged(prompt_text, pid, arm, beta, need_q1=False, max_retries=2, max_new_tokens=1500):
    """Returns a list of attempt-log rows (one per generation attempt, success or not)."""
    logs = []
    for attempt in range(1, max_retries + 1):
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
        raw_text = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        claude_raw, parsed = parse_with_claude(raw_text, need_q1=need_q1)

        logs.append({
            "pid": pid,
            "arm": arm,
            "beta": beta,
            "attempt": attempt,
            "llama_raw": raw_text,
            "claude_raw": claude_raw,
            "success": parsed is not None,
            "manip_check": parsed["Q2"] if parsed else None,
            "elevation": parsed["Q3"] if parsed else None,
            "dv1": parsed["Q4"] if parsed else None,
            "q1_writing": parsed.get("Q1") if (parsed and need_q1) else None,
        })
        if parsed is not None:
            break  # 成功了就不用再重试

    logs[-1]["is_final"] = True
    for row in logs[:-1]:
        row["is_final"] = False
    return logs

## 9. Steering hook (registered on the exact PsySET hook point, plain PyTorch API so it reliably fires on every generation step)

In [ ]:
class SteeringContext:
    """layer_vectors: {layer_idx: 该层自己的归一化向量}. 支持同时在多个层上挂钩子，
    每层各自用该层自己提取出来的方向，不是把一层的向量硬塞到别的层上。"""

    def __init__(self, model, layer_vectors, coeff):
        self.handles = []
        self.model = model
        self.layer_vectors = layer_vectors
        self.coeff = coeff

    def _make_hook(self, vector):
        def hook_fn(module, inputs, output):
            return output + self.coeff * vector
        return hook_fn

    def __enter__(self):
        for layer_idx, vector in self.layer_vectors.items():
            module = self.model.model.layers[layer_idx].hook_after_mlp_hs
            handle = module.register_forward_hook(self._make_hook(vector))
            self.handles.append(handle)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        for handle in self.handles:
            handle.remove()

## 10. Fetch personas live from the Twin-2K-500 dataset

In [ ]:
import requests


def fetch_personas(n=5, offset=0):
    url = (
        "https://datasets-server.huggingface.co/rows"
        f"?dataset=LLM-Digital-Twin%2FTwin-2K-500&config=full_persona&split=data"
        f"&offset={offset}&length={n}"
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    rows = resp.json()["rows"]
    return [(r["row"]["pid"], r["row"]["persona_summary"]) for r in rows]

## 11. Simulation arm (real prompt, no injection) and Injection arm (skip Q1, steer)

In [ ]:
def build_downstream_block():
    return "Q2:\n" + Q_MANIP_CHECK + "\n\nQ3:\n" + Q_FATIGUE + "\n\nQ4:\n" + Q_PREDICTABILITY


def run_simulation_arm_logged(pid, persona_text, cond):
    instruction = QID31_PROMPT if cond == "prime" else QID32_PROMPT
    questions_block = "Q1:\n" + Q_WRITING.format(instruction=instruction) + "\n\n" + build_downstream_block()
    user_content = (
        COMBINED_PROMPT_HEADER + persona_text + COMBINED_PROMPT_SEPARATOR
        + GENERAL_LLM_INSTRUCTION + questions_block
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    arm_name = "real_prime" if cond == "prime" else "real_control"
    return generate_and_extract_logged(prompt_text, pid=pid, arm=arm_name, beta=None, need_q1=True)


def run_injection_arm_logged(pid, persona_text, beta):
    questions_block = build_downstream_block()
    user_content = (
        COMBINED_PROMPT_HEADER + persona_text + COMBINED_PROMPT_SEPARATOR
        + GENERAL_LLM_INSTRUCTION + questions_block
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    arm_name = "inject_lackofcontrol" if beta > 0 else "inject_incontrol"

    with SteeringContext(model, LAYER_VECTORS, beta):
        logs = generate_and_extract_logged(prompt_text, pid=pid, arm=arm_name, beta=abs(beta), need_q1=False)
    return logs

## 12. Calibrate β on a small subset (match Arm-1's manipulation-check average), then run the full 4-arm comparison
`BETA` is fixed after calibration and reused as-is for fatigue / predictability (do not re-tune per measure).

In [ ]:
BETA_RANGE = [2.0,4.5]
N_CALIB_PERSONAS = 3

calib_personas = fetch_personas(n=N_CALIB_PERSONAS, offset=0)

for beta_try in BETA_RANGE:
    scores = []
    failed = False
    for pid, persona_text in calib_personas:
        logs = run_injection_arm_logged(pid, persona_text, beta=beta_try)
        final = logs[-1]
        if final["success"]:
            scores.append(sum(final["manip_check"]) / 3)
        else:
            failed = True
            break

    if scores:
        if failed:
            print(f"beta={beta_try}: 前 {len(scores)} 个persona成功, mean manip_check = {sum(scores)/len(scores):.2f}, 但随后出现失败")
        else:
            print(f"beta={beta_try}: mean manip_check = {sum(scores)/len(scores):.2f} (n={len(scores)}, target: ~6)")
    else:
        print(f"beta={beta_try}: 第一个persona就失败了")

    if failed:
        print(f"beta={beta_try}: 出现失败, 停止整个β扫描")
        break

    # quick text-quality eyeball check
    pid0, persona_text0 = calib_personas[0]
    questions_block = build_downstream_block()
    user_content = (
        COMBINED_PROMPT_HEADER + persona_text0 + COMBINED_PROMPT_SEPARATOR
        + GENERAL_LLM_INSTRUCTION + questions_block
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with SteeringContext(model, LAYER_VECTORS, beta_try):
        with torch.no_grad():
            output_ids = model.generate(
                inputs.input_ids, attention_mask=inputs.attention_mask,
                max_new_tokens=300, do_sample=True, temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    sample_text = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"  [text quality check @ beta={beta_try}]: {sample_text[:200]!r}")
    print()

In [ ]:
from tqdm.auto import tqdm

N_PERSONAS = 100
BETA_VALUES_LACKOFCONTROL = [0.0, 1.8, 2.5, 3.2, 3.9]
BETA_VALUES_INCONTROL = [2.5]

personas = fetch_personas(n=N_PERSONAS, offset=0)
all_logs = []

total_combos_per_persona = 2 + len(BETA_VALUES_LACKOFCONTROL) + len(BETA_VALUES_INCONTROL)
combo_count = 0

for i, (pid, persona_text) in enumerate(tqdm(personas, desc="Personas")):
    print(f"\n--- persona {i+1}/{len(personas)} (pid={pid}) ---")

    logs = run_simulation_arm_logged(pid, persona_text, cond="prime")
    all_logs.extend(logs)
    combo_count += 1
    print(f"  [{combo_count}] real_prime done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    logs = run_simulation_arm_logged(pid, persona_text, cond="control")
    all_logs.extend(logs)
    combo_count += 1
    print(f"  [{combo_count}] real_control done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    for beta in BETA_VALUES_LACKOFCONTROL:
        logs = run_injection_arm_logged(pid, persona_text, beta=+beta)
        all_logs.extend(logs)
        combo_count += 1
        print(f"  [{combo_count}] inject_lackofcontrol beta={beta} done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    for beta in BETA_VALUES_INCONTROL:
        logs = run_injection_arm_logged(pid, persona_text, beta=-beta)
        all_logs.extend(logs)
        combo_count += 1
        print(f"  [{combo_count}] inject_incontrol beta={beta} done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    print(f"pid={pid} done ({combo_count}/{total_combos_per_persona * len(personas)} 个组合已完成)")

results_df = pd.DataFrame(all_logs)
results_df.to_csv("affective_priming_control_full_log.csv", index=False)
results_df


## 13. Summary: mean scores per arm

In [ ]:
def mean_of_list_col(series):
    return series.apply(lambda x: sum(x) / len(x)).mean()


final_df = results_df[(results_df["is_final"]) & (results_df["success"])]

summary = final_df.groupby(["arm", "beta"], dropna=False).agg(
    n=("pid", "count"),
    manip_check_mean=("manip_check", mean_of_list_col),
    elevation_mean=("elevation", mean_of_list_col),
    dv1_mean=("dv1", mean_of_list_col),
)
print(summary)

# 顺带看一下每个组合的失败率，现在数据够细可以直接算出来
fail_rate = results_df[results_df["is_final"]].groupby(["arm", "beta"], dropna=False)["success"].agg(
    n_total="count", n_success="sum"
)
fail_rate["fail_rate"] = 1 - fail_rate["n_success"] / fail_rate["n_total"]
print()
print(fail_rate)

from google.colab import files
files.download("affective_priming_control_full_log.csv")